# Phase 3 — Post-Finetuning Attention Analysis

**Fully self-contained notebook.** No GitHub clone required.

Compares pretrained vs. fine-tuned checkpoint attention patterns for:
- Hindi and Nepali
- Early layer (Layer 0) vs. Late layer (Layer 7)
- All 7 attention heads
- Familiar vs. held-out reasoning templates

Uses the **same metric definitions** as Phase 2 (Mean Attention Distance, Entropy).

**You need to provide (attach to Kaggle notebook):**
- Phase 2 pretrained checkpoints for Hindi and Nepali (`step_00020000.pt`)
- Phase 3 fine-tuned checkpoints for Hindi and Nepali (`step_00003000.pt`)
- Tokenizer `.model` files for both languages
- Test JSONL files for both languages

**Edit Section 1 paths before running.**

> ⛔ `RUN_ANALYSIS` defaults to `False`. Setup cells are safe to run without launching the full analysis.

## Section 1 — Configuration (edit paths here)

In [ ]:
# ================================================================
# CONFIGURATION — edit this cell only before running on Kaggle
# ================================================================

# ── Hindi checkpoints ─────────────────────────────────────────────
HINDI_PRETRAINED_CKPT = "/kaggle/input/notebooks/negative00/notebookb7b29f6528/phase2_outputs/hindi_checkpoints/step_00020000.pt"
HINDI_FINETUNED_CKPT  = "/kaggle/working/results/phase3_finetune/hindi/step_00003000.pt"

# ── Nepali checkpoints ────────────────────────────────────────────
NEPALI_PRETRAINED_CKPT = "/kaggle/input/notebooks/negative00/notebookb7b29f6528/phase2_outputs/nepali_checkpoints/step_00020000.pt"
NEPALI_FINETUNED_CKPT  = "/kaggle/working/results/phase3_finetune/nepali/step_00003000.pt"

# ── Tokenizers ───────────────────────────────────────────────────
HINDI_TOKENIZER  = "/kaggle/input/datasets/negative00/phase2-cloud-data/hindi/tokenizer/models/hindi_bpe_10k_final.model"
NEPALI_TOKENIZER = "/kaggle/input/datasets/negative00/phase2-cloud-data/nepali/tokenizer/models/nepali_bpe_10k_assignment.model"

# ── Test JSONL files ─────────────────────────────────────────────
HINDI_TEST  = "/kaggle/input/datasets/negative00/phase-3-finetune-dataset/hindi_test.jsonl"
NEPALI_TEST = "/kaggle/input/datasets/negative00/phase-3-finetune-dataset/nepali_test.jsonl"

# ── Output directory ─────────────────────────────────────────────
OUTPUT_DIR = "/kaggle/working/results/phase3_attention"

# ── Analysis settings ─────────────────────────────────────────────
# Number of examples to select per category (familiar / held-out) per language
N_EXAMPLES_PER_CATEGORY = 3

# Layers to analyse (0 = early, 7 = late; consistent with Phase 2 convention)
EARLY_LAYER = 0
LATE_LAYER  = 7

# Head used for heatmap visualisation (Head 0 for Phase 2 continuity)
HEATMAP_HEAD = 0

# Max tokens for sequence truncation in heatmaps (for plot readability)
HEATMAP_MAX_TOKENS = 40

# ── Safety flag ───────────────────────────────────────────────────
# Set to True ONLY when ready to run the full analysis on Kaggle
RUN_ANALYSIS = False

# ── Phase 2 reference attention values (from results/phase2_evaluation/evaluation_summary.json)
# Used for contextual comparison in the summary section.
PHASE2_REFERENCE = {
    "hindi":  {"attention_entropy": 1.7901, "mean_attention_distance": 8.1937},
    "nepali": {"attention_entropy": 1.7572, "mean_attention_distance": 8.0824},
}

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "figures"), exist_ok=True)

print("Configuration loaded.")
print(f"  Hindi pretrained  : {HINDI_PRETRAINED_CKPT}")
print(f"  Hindi finetuned   : {HINDI_FINETUNED_CKPT}")
print(f"  Nepali pretrained : {NEPALI_PRETRAINED_CKPT}")
print(f"  Nepali finetuned  : {NEPALI_FINETUNED_CKPT}")
print(f"  Output dir        : {OUTPUT_DIR}")
print(f"  RUN_ANALYSIS      : {RUN_ANALYSIS}")

## Section 2 — Environment Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "sentencepiece", "-q"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "seaborn", "-q"], check=True)

import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device   : {DEVICE}")

## Section 3 — Model Architecture (Phase 2, inlined)

Exact copy of `phase2/src/model.py` with `return_attention=True` support.
**Do not modify this cell.**

In [ ]:
# ── LMConfig (frozen Phase 2 spec) ───────────────────────────────
import math
import torch
import torch.nn as nn
from dataclasses import dataclass

@dataclass
class LMConfig:
    vocab_size:  int   = 10000
    d_model:     int   = 448
    n_layers:    int   = 8
    n_heads:     int   = 7
    d_ff:        int   = 2048
    max_seq_len: int   = 512
    dropout:     float = 0.1
    bias:        bool  = True


class InputEmbeddings(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.max_seq_len        = config.max_seq_len
        self.token_embedding    = nn.Embedding(config.vocab_size, config.d_model)
        self.position_embedding = nn.Embedding(config.max_seq_len, config.d_model)
        self.dropout            = nn.Dropout(config.dropout)

    def forward(self, input_ids):
        B, T = input_ids.shape
        if T > self.max_seq_len:
            raise ValueError(f"Sequence length {T} exceeds max_seq_len {self.max_seq_len}")
        device      = input_ids.device
        position_ids = torch.arange(0, T, dtype=torch.long, device=device)
        x = self.token_embedding(input_ids) + self.position_embedding(position_ids)
        return self.dropout(x)


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.d_model % config.n_heads == 0
        self.d_model     = config.d_model
        self.n_heads     = config.n_heads
        self.head_dim    = config.d_model // config.n_heads
        self.max_seq_len = config.max_seq_len
        self.q_proj      = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.k_proj      = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.v_proj      = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.out_proj    = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.attn_dropout  = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        mask = torch.tril(torch.ones(config.max_seq_len, config.max_seq_len)).view(
            1, 1, config.max_seq_len, config.max_seq_len)
        self.register_buffer("bias_mask", mask)

    def forward(self, x, return_attention=False):
        B, T, C = x.size()
        q = self.q_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(self.head_dim))
        att = att.masked_fill(self.bias_mask[:, :, :T, :T] == 0, float('-inf'))
        att = torch.nn.functional.softmax(att, dim=-1)
        att_d = self.attn_dropout(att)
        y = (att_d @ v).transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.out_proj(y))
        if return_attention:
            return y, att   # att is BEFORE dropout for analysis
        return y


class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.d_model, config.d_ff, bias=config.bias),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.d_ff, config.d_model, bias=config.bias),
            nn.Dropout(config.dropout),
        )
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.d_model)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.d_model)
        self.ffn  = FeedForward(config)

    def forward(self, x, return_attention=False):
        # Pre-LayerNorm: x = x + Attention(LayerNorm(x))
        if return_attention:
            attn_out, att_weights = self.attn(self.ln_1(x), return_attention=True)
            x = x + attn_out
        else:
            x = x + self.attn(self.ln_1(x))
        # Pre-LayerNorm: x = x + FFN(LayerNorm(x))
        x = x + self.ffn(self.ln_2(x))
        if return_attention:
            return x, att_weights
        return x


class DecoderOnlyLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config    = config
        self.embedding = InputEmbeddings(config)
        self.blocks    = nn.ModuleList([TransformerBlock(config) for _ in range(config.n_layers)])
        self.ln_f      = nn.LayerNorm(config.d_model)
        self.lm_head   = nn.Linear(config.d_model, config.vocab_size, bias=config.bias)
        self.lm_head.weight = self.embedding.token_embedding.weight  # weight tying

    def forward(self, input_ids, return_attention=False):
        x          = self.embedding(input_ids)
        attentions = [] if return_attention else None
        for block in self.blocks:
            if return_attention:
                x, att = block(x, return_attention=True)
                attentions.append(att)
            else:
                x = block(x)
        x      = self.ln_f(x)
        logits = self.lm_head(x)
        if return_attention:
            return logits, attentions   # attentions: list of [B, n_heads, T, T]
        return logits


# ── Sanity check ─────────────────────────────────────────────────
_cfg = LMConfig()
_tmp = DecoderOnlyLM(_cfg)
_n   = sum(p.numel() for p in _tmp.parameters())
assert _n == 25_871_504, f"Unexpected parameter count: {_n}"
assert _cfg.n_layers  == 8,  "Expected 8 layers"
assert _cfg.n_heads   == 7,  "Expected 7 heads"
assert EARLY_LAYER    == 0,  "EARLY_LAYER must be 0"
assert LATE_LAYER     == 7,  "LATE_LAYER must be 7 (last layer index for 8-layer model)"
print(f"DecoderOnlyLM defined. Parameters: {_n:,}")
print(f"Layers: {_cfg.n_layers}  Heads: {_cfg.n_heads}  Early: {EARLY_LAYER}  Late: {LATE_LAYER}")
del _tmp

## Section 4 — Checkpoint Loader

In [ ]:
# ── Checkpoint loader (matches phase2/src/trainer.py::load_checkpoint) ────
import torch
from pathlib import Path

CHECKPOINT_VERSION = 1

def load_checkpoint(checkpoint_path, model, optimizer=None, scheduler=None,
                    scaler=None, device=None, restore_rng=False):
    """Load a Phase 2/3 checkpoint into model. Returns payload dict."""
    if device is None:
        device = torch.device("cpu")
    payload = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if payload.get("version") != CHECKPOINT_VERSION:
        raise ValueError(
            f"Checkpoint version mismatch. Expected {CHECKPOINT_VERSION}, got {payload.get('version')}"
        )
    model.load_state_dict(payload["model_state"])
    if optimizer is not None:
        optimizer.load_state_dict(payload["optimizer_state"])
    if scheduler is not None:
        scheduler.load_state_dict(payload["scheduler_state"])
    if scaler is not None and "scaler_state" in payload:
        scaler.load_state_dict(payload["scaler_state"])
    return payload


def load_model_from_ckpt(ckpt_path, device):
    """Instantiate DecoderOnlyLM and load weights from checkpoint."""
    model = DecoderOnlyLM(LMConfig()).to(device)
    payload = load_checkpoint(
        ckpt_path, model,
        device=torch.device(device),
        restore_rng=False,
    )
    model.eval()
    step = payload.get("step", "unknown")
    print(f"  Loaded: {ckpt_path}  (step={step})")
    return model, step


print("Checkpoint loader defined.")

## Section 5 — Example Selection Helpers

In [ ]:
import json
import re
import sentencepiece as spm

# ── UNSEEN_TEMPLATE_IDS — authoritative source of truth ──────────
# Copied verbatim from phase3/scripts/validate_dataset.py and
# phase3/scripts/evaluate_reasoning.py
UNSEEN_TEMPLATE_IDS: frozenset = frozenset({
    'H3', 'H7', 'H9', 'H22', 'H31', 'H40',   # Hindi held-out templates
    'N3', 'N7', 'N9', 'N22', 'N31', 'N40',   # Nepali held-out templates
})


def is_held_out(example: dict) -> bool:
    """True iff any constituent template_id is in UNSEEN_TEMPLATE_IDS."""
    return any(
        t in UNSEEN_TEMPLATE_IDS
        for t in example.get('template_id', '').split('+')
    )


def load_jsonl(path: str) -> list:
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data


def build_prompt(example: dict) -> str:
    """Construct the Phase 3 prompt format — identical to evaluate_reasoning.py."""
    return f"\u092a\u094d\u0930\u0936\u094d\u0928: {example['premise']} {example['question']} \u0909\u0924\u094d\u0924\u0930: "
    # Decoded: "प्रश्न: {premise} {question} उत्तर: "


def select_examples(dataset: list, n_per_category: int, lang: str) -> dict:
    """
    Deterministically select up to n_per_category examples from the
    FAMILIAR and HELD_OUT categories of the test set.

    Selection procedure:
    - Iterate the JSONL in order (deterministic — no shuffling).
    - Take the first n_per_category examples of each category.
    - Record example ID, template ID, task type, attribute, and split
      for full reproducibility.

    Returns dict with keys 'familiar' and 'held_out', each a list of examples.
    """
    familiar  = []
    held_out  = []

    for ex in dataset:
        # Validate the example is from the test split
        assert ex.get('split') == 'test', (
            f"Example {ex.get('id')} has split={ex.get('split')}; expected 'test'"
        )
        if is_held_out(ex) and len(held_out) < n_per_category:
            held_out.append(ex)
        elif not is_held_out(ex) and len(familiar) < n_per_category:
            familiar.append(ex)
        if len(familiar) >= n_per_category and len(held_out) >= n_per_category:
            break

    print(f"  {lang}: selected {len(familiar)} familiar, {len(held_out)} held-out examples")
    for ex in familiar:
        print(f"    [familiar]  id={ex.get('id')}  template={ex.get('template_id')}  "
              f"task={ex.get('task_type')}  attr={ex.get('attribute')}")
    for ex in held_out:
        print(f"    [held_out]  id={ex.get('id')}  template={ex.get('template_id')}  "
              f"task={ex.get('task_type')}  attr={ex.get('attribute')}")

    return {'familiar': familiar, 'held_out': held_out}


print("Example selection helpers defined.")
print(f"UNSEEN_TEMPLATE_IDS = {sorted(UNSEEN_TEMPLATE_IDS)}")

## Section 6 — Attention Metric Helpers (Phase 2 definitions, reused verbatim)

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive backend safe for Kaggle
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path


def calc_attn_stats(attn_matrix_np: np.ndarray):
    """
    Compute attention metrics from a per-layer attention tensor.

    Inputs
    ------
    attn_matrix_np : np.ndarray of shape (n_heads, seq_len, seq_len)
        Softmax attention probabilities (post-causal-mask, pre-dropout).

    Returns
    -------
    per_head : list of dict, one entry per head containing:
        - 'mean_distance' : mean attention distance (local vs. long-range)
        - 'entropy'       : mean attention entropy (sharpness)
    mean_distance_all : float  — averaged across all heads
    entropy_all       : float  — averaged across all heads

    Phase 2 definitions (preserved exactly)
    ----------------------------------------
    Mean Attention Distance = sum_j ( p(i,j) * |i - j| ),  averaged over
        all query positions i and all heads.
    Entropy = -sum_j ( p(i,j) * log(p(i,j) + eps) ),  averaged over i and heads.

    A higher Mean Attention Distance indicates more long-range attention.
    A lower Mean Attention Distance indicates more local/short-range attention.
    A higher Entropy indicates more diffuse/dispersed attention.
    A lower Entropy indicates more peaked/focused attention.
    """
    n_heads, seq_len, _ = attn_matrix_np.shape
    epsilon = 1e-10

    indices     = np.arange(seq_len)
    dist_matrix = np.abs(indices.reshape(-1, 1) - indices.reshape(1, -1))  # (seq, seq)

    per_head = []
    for h in range(n_heads):
        head_attn = attn_matrix_np[h]                        # (seq, seq)
        # Entropy per query position, then mean
        ent_per_pos    = -np.sum(head_attn * np.log(head_attn + epsilon), axis=-1)  # (seq,)
        mean_entropy   = float(np.mean(ent_per_pos))
        # Distance per query position, then mean
        dist_per_pos   = np.sum(head_attn * dist_matrix, axis=-1)                   # (seq,)
        mean_distance  = float(np.mean(dist_per_pos))
        per_head.append({'head': h, 'mean_distance': mean_distance, 'entropy': mean_entropy})

    mean_distance_all = float(np.mean([h['mean_distance'] for h in per_head]))
    entropy_all       = float(np.mean([h['entropy']       for h in per_head]))

    return per_head, mean_distance_all, entropy_all


@torch.inference_mode()
def extract_attention(model, sp, prompt_text, device):
    """
    Tokenize prompt and run model with return_attention=True.

    Returns
    -------
    attentions   : list of np.ndarray, one per layer, shape (n_heads, T, T)
    token_labels : list of str — decoded surface form of each token
    token_ids    : list of int
    """
    model.eval()
    ids = sp.Encode(prompt_text, out_type=int)
    ids = ids[:HEATMAP_MAX_TOKENS]   # truncate for plot readability
    token_labels = [sp.IdToPiece(i) for i in ids]

    x = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)  # (1, T)
    _, attn_list = model(x, return_attention=True)

    # attn_list: list of Tensors [1, n_heads, T, T]  — one per layer
    # Detach and move to numpy immediately to free GPU memory
    attentions = [a[0].detach().cpu().numpy() for a in attn_list]   # (n_heads, T, T) each
    return attentions, token_labels, ids


def validate_attention_shape(attentions, expected_layers=8, expected_heads=7):
    """Assert attention output has the expected shape."""
    assert len(attentions) == expected_layers, (
        f"Expected {expected_layers} layers, got {len(attentions)}"
    )
    n_heads, T1, T2 = attentions[0].shape
    assert n_heads == expected_heads, (
        f"Expected {expected_heads} heads, got {n_heads}"
    )
    assert T1 == T2, f"Attention matrix is not square: ({T1}, {T2})"
    assert attentions[EARLY_LAYER] is not None, f"Layer {EARLY_LAYER} is missing"
    assert attentions[LATE_LAYER]  is not None, f"Layer {LATE_LAYER} is missing"
    return n_heads, T1


def plot_attention_heatmap(
    attn_matrix_np, token_labels, title, save_path,
    vmin=0.0, vmax=None
):
    """
    Plot a single-head attention heatmap using the Phase 2 seaborn/viridis style.

    attn_matrix_np : np.ndarray of shape (T, T)  — one head's attention
    token_labels   : list of str, length T
    title          : str
    save_path      : str or Path
    vmin, vmax     : colour scale bounds (set vmax consistently across paired plots)
    """
    T = attn_matrix_np.shape[0]
    fig_size = max(6, T * 0.35)
    plt.figure(figsize=(fig_size, fig_size * 0.85))
    ax = sns.heatmap(
        attn_matrix_np,
        xticklabels=token_labels,
        yticklabels=token_labels,
        cmap='viridis',
        vmin=vmin,
        vmax=vmax,
        square=True,
    )
    plt.title(title, fontsize=10, pad=8)
    plt.xlabel("Key Token",   fontsize=9)
    plt.ylabel("Query Token", fontsize=9)
    ax.set_xticklabels(ax.get_xticklabels(), fontsize=6, rotation=90)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=6, rotation=0)
    plt.tight_layout()
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {save_path}")


print("Attention metric helpers defined.")

## Section 7 — Environment / Data Validation

Run this section before the full analysis to confirm everything is in place.

In [ ]:
from pathlib import Path

print("=" * 60)
print("PRE-FLIGHT VALIDATION")
print("=" * 60)

errors = []

REQUIRED_FILES = {
    "Hindi pretrained ckpt":  HINDI_PRETRAINED_CKPT,
    "Hindi finetuned ckpt":   HINDI_FINETUNED_CKPT,
    "Nepali pretrained ckpt": NEPALI_PRETRAINED_CKPT,
    "Nepali finetuned ckpt":  NEPALI_FINETUNED_CKPT,
    "Hindi tokenizer":        HINDI_TOKENIZER,
    "Nepali tokenizer":       NEPALI_TOKENIZER,
    "Hindi test JSONL":       HINDI_TEST,
    "Nepali test JSONL":      NEPALI_TEST,
}

for label, fpath in REQUIRED_FILES.items():
    exists = Path(fpath).exists()
    status = "✓" if exists else "✗ MISSING"
    print(f"  {status}  {label}: {fpath}")
    if not exists:
        errors.append(f"{label} not found: {fpath}")

print()

# ── Tokenizer load check ──────────────────────────────────────────
try:
    _sp_h = spm.SentencePieceProcessor()
    _sp_h.Load(HINDI_TOKENIZER)
    assert _sp_h.GetPieceSize() == 10000, f"Hindi vocab size {_sp_h.GetPieceSize()} != 10000"
    print(f"  ✓  Hindi tokenizer loaded (vocab={_sp_h.GetPieceSize()})")
except Exception as e:
    errors.append(f"Hindi tokenizer load failed: {e}")
    print(f"  ✗  Hindi tokenizer: {e}")

try:
    _sp_n = spm.SentencePieceProcessor()
    _sp_n.Load(NEPALI_TOKENIZER)
    assert _sp_n.GetPieceSize() == 10000, f"Nepali vocab size {_sp_n.GetPieceSize()} != 10000"
    print(f"  ✓  Nepali tokenizer loaded (vocab={_sp_n.GetPieceSize()})")
except Exception as e:
    errors.append(f"Nepali tokenizer load failed: {e}")
    print(f"  ✗  Nepali tokenizer: {e}")

print()

# ── Dataset existence and split check ────────────────────────────
for lang, path in [("hindi", HINDI_TEST), ("nepali", NEPALI_TEST)]:
    try:
        ds = load_jsonl(path)
        bad_splits = [d for d in ds if d.get('split') != 'test']
        fam  = sum(1 for d in ds if not is_held_out(d))
        held = sum(1 for d in ds if     is_held_out(d))
        print(f"  {lang} test: total={len(ds)}  familiar={fam}  held-out={held}")
        if bad_splits:
            errors.append(f"{lang} test: {len(bad_splits)} examples with split != 'test'")
    except Exception as e:
        errors.append(f"{lang} test JSONL load failed: {e}")
        print(f"  ✗  {lang} test: {e}")

print()

if errors:
    print(f"[FAIL] {len(errors)} pre-flight error(s):")
    for e in errors:
        print(f"  - {e}")
    print("\nFix the above before setting RUN_ANALYSIS = True.")
else:
    print("[PASS] All pre-flight checks passed. Safe to set RUN_ANALYSIS = True.")

## Section 8 — Example Selection (deterministic, from TEST split)

In [ ]:
if not RUN_ANALYSIS:
    print("RUN_ANALYSIS=False — skipping example selection.")
    print("Set RUN_ANALYSIS = True in Section 1 to proceed.")
else:
    print("Loading test datasets and selecting examples...")

    hindi_test  = load_jsonl(HINDI_TEST)
    nepali_test = load_jsonl(NEPALI_TEST)

    assert all(d.get('split') == 'test' for d in hindi_test),  "hindi_test: non-test examples found"
    assert all(d.get('split') == 'test' for d in nepali_test), "nepali_test: non-test examples found"

    print("\nHindi:")
    hindi_selected  = select_examples(hindi_test,  N_EXAMPLES_PER_CATEGORY, "hindi")
    print("\nNepali:")
    nepali_selected = select_examples(nepali_test, N_EXAMPLES_PER_CATEGORY, "nepali")

    # Validate held-out classification is consistent with UNSEEN_TEMPLATE_IDS
    for lang, sel in [("hindi", hindi_selected), ("nepali", nepali_selected)]:
        for ex in sel['familiar']:
            assert not is_held_out(ex), (
                f"{lang} example {ex.get('id')} classified as familiar but has held-out template"
            )
        for ex in sel['held_out']:
            assert is_held_out(ex), (
                f"{lang} example {ex.get('id')} classified as held-out but template not in UNSEEN_TEMPLATE_IDS"
            )
    print("\nHeld-out classification assertions passed.")

## Section 9 — Full Attention Analysis

For each language: load pretrained → extract attention → load finetuned → extract attention.
Paired token IDs are verified to be identical before comparing.

In [ ]:
if not RUN_ANALYSIS:
    print("RUN_ANALYSIS=False — skipping full analysis.")
else:
    import gc

    # ── Results accumulator ───────────────────────────────────────
    all_metrics = {}   # keyed by (lang, model_type, layer_name, category, example_idx)
    metadata    = []   # list of per-example metadata records

    RUNS = [
        ("hindi",  HINDI_PRETRAINED_CKPT,  HINDI_TOKENIZER,  hindi_selected,  "pretrained", 20000),
        ("hindi",  HINDI_FINETUNED_CKPT,   HINDI_TOKENIZER,  hindi_selected,  "finetuned",  3000),
        ("nepali", NEPALI_PRETRAINED_CKPT,  NEPALI_TOKENIZER, nepali_selected, "pretrained", 20000),
        ("nepali", NEPALI_FINETUNED_CKPT,   NEPALI_TOKENIZER, nepali_selected, "finetuned",  3000),
    ]

    for lang, ckpt_path, tok_path, selected, model_type, expected_step in RUNS:
        print(f"\n{'='*65}")
        print(f"Processing: {lang.upper()} / {model_type.upper()}")
        print(f"Checkpoint : {ckpt_path}")

        # ── Load model ───────────────────────────────────────────
        assert Path(ckpt_path).exists(), f"Checkpoint not found: {ckpt_path}"
        model, step = load_model_from_ckpt(ckpt_path, DEVICE)
        print(f"  Checkpoint step: {step}")

        # ── Load tokenizer ────────────────────────────────────────
        sp = spm.SentencePieceProcessor()
        sp.Load(tok_path)
        assert sp.GetPieceSize() == 10000

        # ── Iterate over categories and examples ─────────────────
        for category in ['familiar', 'held_out']:
            examples = selected[category]
            for ex_idx, example in enumerate(examples):
                prompt_text = build_prompt(example)
                ex_id       = example.get('id', f"{lang}_{category}_{ex_idx}")

                # Extract attention
                attentions, token_labels, token_ids = extract_attention(
                    model, sp, prompt_text, DEVICE
                )

                # Validate shape
                n_heads, seq_len = validate_attention_shape(attentions)
                assert n_heads == 7, f"Expected 7 heads, got {n_heads}"

                # Compute metrics for early and late layers
                for layer_idx, layer_name in [(EARLY_LAYER, 'early'), (LATE_LAYER, 'late')]:
                    attn_np = attentions[layer_idx]   # (n_heads, T, T)
                    per_head, mean_dist_all, entropy_all = calc_attn_stats(attn_np)

                    key = (lang, model_type, layer_name, category, ex_idx)
                    all_metrics[key] = {
                        'lang':          lang,
                        'model_type':    model_type,
                        'layer_name':    layer_name,
                        'layer_idx':     layer_idx,
                        'category':      category,
                        'ex_idx':        ex_idx,
                        'example_id':    ex_id,
                        'template_id':   example.get('template_id', ''),
                        'task_type':     example.get('task_type', ''),
                        'attribute':     example.get('attribute', ''),
                        'question_category': example.get('question_category', ''),
                        'answer':        str(example.get('answer', '')),
                        'seq_len':       seq_len,
                        'mean_distance_all_heads': mean_dist_all,
                        'entropy_all_heads':       entropy_all,
                        'per_head':      per_head,
                        'checkpoint_path': ckpt_path,
                        'step':          step,
                        'tokenizer_path': tok_path,
                    }

                    print(f"  [{category:9s} ex{ex_idx}] Layer {layer_idx} ({layer_name:5s}): "
                          f"mean_dist={mean_dist_all:.4f}  entropy={entropy_all:.4f}")

                # ── Heatmaps (Head 0, for Phase 2 continuity) ────
                for layer_idx, layer_name in [(EARLY_LAYER, 'early'), (LATE_LAYER, 'late')]:
                    if category == 'familiar' and ex_idx == 0:
                        # Only generate heatmaps for the first familiar example
                        # to keep output files manageable
                        attn_head0 = attentions[layer_idx][HEATMAP_HEAD]  # (T, T)
                        fname = f"{lang}_{layer_name}_{model_type}.png"
                        save_path = Path(OUTPUT_DIR) / "figures" / fname
                        title = (
                            f"{lang.capitalize()} — Layer {layer_idx} ({layer_name}) "
                            f"Head {HEATMAP_HEAD} — {model_type.capitalize()}\n"
                            f"ex={ex_id}  template={example.get('template_id')}  "
                            f"seq_len={seq_len}"
                        )
                        plot_attention_heatmap(
                            attn_head0, token_labels, title, save_path
                        )

                # ── Record metadata ───────────────────────────────
                if layer_name == 'late':  # avoid duplicate metadata entries
                    metadata.append({
                        'lang':        lang,
                        'model_type':  model_type,
                        'step':        step,
                        'checkpoint':  ckpt_path,
                        'tokenizer':   tok_path,
                        'category':    category,
                        'ex_idx':      ex_idx,
                        'example_id':  ex_id,
                        'template_id': example.get('template_id', ''),
                        'task_type':   example.get('task_type', ''),
                        'attribute':   example.get('attribute', ''),
                        'question_category': example.get('question_category', ''),
                        'premise':     example.get('premise', ''),
                        'question':    example.get('question', ''),
                        'answer':      str(example.get('answer', '')),
                        'seq_len':     seq_len,
                        'prompt_token_ids': token_ids,
                    })

        # Free GPU memory between runs
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    print("\nAll four model runs complete.")

## Section 10 — Token ID Parity Check

Asserts that pretrained and fine-tuned models received **identical token IDs**
for each paired example, ensuring the comparison is fair.

In [ ]:
if not RUN_ANALYSIS:
    print("RUN_ANALYSIS=False — skipping parity check.")
else:
    print("Checking token ID parity between pretrained and fine-tuned...")

    # Group metadata by (lang, category, ex_idx) to compare pretrained vs finetuned
    from collections import defaultdict
    grouped = defaultdict(dict)
    for record in metadata:
        key = (record['lang'], record['category'], record['ex_idx'])
        grouped[key][record['model_type']] = record

    all_parity_ok = True
    for key, models in grouped.items():
        lang, category, ex_idx = key
        if 'pretrained' in models and 'finetuned' in models:
            ids_pre = models['pretrained']['prompt_token_ids']
            ids_ft  = models['finetuned']['prompt_token_ids']
            if ids_pre != ids_ft:
                print(f"  ✗ MISMATCH: {lang}/{category}/ex{ex_idx}")
                print(f"    pretrained ids: {ids_pre}")
                print(f"    finetuned  ids: {ids_ft}")
                all_parity_ok = False
            else:
                print(f"  ✓  {lang}/{category}/ex{ex_idx} — token IDs match (len={len(ids_pre)})")

    assert all_parity_ok, "Token ID parity check FAILED — pretrained and finetuned saw different inputs!"
    print("\nToken ID parity: PASSED")

## Section 11 — Save Outputs

In [ ]:
if not RUN_ANALYSIS:
    print("RUN_ANALYSIS=False — skipping output save.")
else:
    import csv
    import json
    from pathlib import Path

    out = Path(OUTPUT_DIR)

    # ── 1. Flat metrics list (per example × layer × head) ─────────
    flat_rows = []
    for key, m in all_metrics.items():
        for h_data in m['per_head']:
            flat_rows.append({
                'lang':               m['lang'],
                'model_type':         m['model_type'],
                'layer_name':         m['layer_name'],
                'layer_idx':          m['layer_idx'],
                'head':               h_data['head'],
                'category':           m['category'],
                'ex_idx':             m['ex_idx'],
                'example_id':         m['example_id'],
                'template_id':        m['template_id'],
                'task_type':          m['task_type'],
                'attribute':          m['attribute'],
                'question_category':  m['question_category'],
                'answer':             m['answer'],
                'seq_len':            m['seq_len'],
                'mean_distance':      h_data['mean_distance'],
                'entropy':            h_data['entropy'],
                'mean_distance_all_heads': m['mean_distance_all_heads'],
                'entropy_all_heads':       m['entropy_all_heads'],
                'step':               m['step'],
                'checkpoint_path':    m['checkpoint_path'],
                'tokenizer_path':     m['tokenizer_path'],
            })

    # ── 2. JSON ───────────────────────────────────────────────────
    # Serialise all_metrics but strip per_head list from nested dicts
    json_payload = []
    for key, m in all_metrics.items():
        entry = {k: v for k, v in m.items() if k != 'per_head'}
        entry['per_head'] = m['per_head']
        json_payload.append(entry)

    json_path = out / "attention_metrics.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(json_payload, f, indent=2, ensure_ascii=False)
    print(f"Saved: {json_path} ({json_path.stat().st_size} bytes)")

    # ── 3. CSV ────────────────────────────────────────────────────
    csv_path = out / "attention_metrics.csv"
    fieldnames = list(flat_rows[0].keys()) if flat_rows else []
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(flat_rows)
    print(f"Saved: {csv_path} ({csv_path.stat().st_size} bytes)")

    # ── 4. Metadata JSON ──────────────────────────────────────────
    meta_path = out / "selected_examples_metadata.json"
    with open(meta_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    print(f"Saved: {meta_path} ({meta_path.stat().st_size} bytes)")

    # ── 5. Assert no raw attention tensors on disk ────────────────
    raw_tensor_files = list(out.rglob("*.npy")) + list(out.rglob("*.npz")) + list(out.rglob("*.pt"))
    assert len(raw_tensor_files) == 0, (
        f"Raw attention tensors found on disk: {raw_tensor_files}"
    )
    print("Assertion: no raw attention tensors saved to disk — PASSED")

    print("\nAll outputs saved.")

## Section 12 — Comparison Summary Tables

In [ ]:
if not RUN_ANALYSIS:
    print("RUN_ANALYSIS=False — skipping summary tables.")
else:
    print("=" * 80)
    print("TABLE 1 — Aggregate: Language × Layer × Model (all heads, all examples)")
    print("=" * 80)
    print(f"{'Language':<8} {'Layer':<6} {'Model':<12} {'Mean Attn Distance':>20} {'Entropy':>10}")
    print("-" * 58)

    from collections import defaultdict

    agg = defaultdict(list)  # (lang, layer_name, model_type) -> list of (mean_dist, entropy)
    for key, m in all_metrics.items():
        agg_key = (m['lang'], m['layer_name'], m['model_type'])
        agg[agg_key].append((m['mean_distance_all_heads'], m['entropy_all_heads']))

    table1 = []
    for lang in ['hindi', 'nepali']:
        for layer_name in ['early', 'late']:
            for model_type in ['pretrained', 'finetuned']:
                vals = agg.get((lang, layer_name, model_type), [])
                if not vals:
                    continue
                import numpy as np
                avg_dist = float(np.mean([v[0] for v in vals]))
                avg_ent  = float(np.mean([v[1] for v in vals]))
                print(f"{lang:<8} {layer_name:<6} {model_type:<12} {avg_dist:>20.4f} {avg_ent:>10.4f}")
                table1.append((lang, layer_name, model_type, avg_dist, avg_ent))

    print()
    print("=" * 80)
    print("TABLE 2 — Head-level: Language × Layer × Head × Pretrained vs Fine-tuned")
    print("=" * 80)
    print(f"{'Lang':<7} {'Layer':<6} {'Head':>4}  "
          f"{'Pre dist':>10} {'FT dist':>9} {'Δ dist':>8}  "
          f"{'Pre ent':>8} {'FT ent':>8} {'Δ ent':>7}")
    print("-" * 70)

    head_agg = defaultdict(lambda: defaultdict(list))  # (lang, layer_name, model_type) -> head -> [dist, ent]
    for key, m in all_metrics.items():
        k = (m['lang'], m['layer_name'], m['model_type'])
        for h_data in m['per_head']:
            head_agg[k][h_data['head']].append((h_data['mean_distance'], h_data['entropy']))

    table2_rows = []
    for lang in ['hindi', 'nepali']:
        for layer_name in ['early', 'late']:
            pre_key = (lang, layer_name, 'pretrained')
            ft_key  = (lang, layer_name, 'finetuned')
            for h in range(7):
                pre_vals = head_agg.get(pre_key, {}).get(h, [])
                ft_vals  = head_agg.get(ft_key,  {}).get(h, [])
                if not pre_vals or not ft_vals:
                    continue
                import numpy as np
                pre_d = float(np.mean([v[0] for v in pre_vals]))
                ft_d  = float(np.mean([v[0] for v in ft_vals]))
                pre_e = float(np.mean([v[1] for v in pre_vals]))
                ft_e  = float(np.mean([v[1] for v in ft_vals]))
                delta_d = ft_d - pre_d
                delta_e = ft_e - pre_e
                print(f"{lang:<7} {layer_name:<6} {h:>4}  "
                      f"{pre_d:>10.4f} {ft_d:>9.4f} {delta_d:>+8.4f}  "
                      f"{pre_e:>8.4f} {ft_e:>8.4f} {delta_e:>+7.4f}")
                table2_rows.append({
                    'lang': lang, 'layer': layer_name, 'head': h,
                    'pretrained_distance': pre_d, 'finetuned_distance': ft_d, 'delta_distance': delta_d,
                    'pretrained_entropy':  pre_e, 'finetuned_entropy':  ft_e, 'delta_entropy':  delta_e,
                })

    print()
    print("=" * 80)
    print("TABLE 3 — Familiar vs Held-out: aggregate mean_distance and entropy")
    print("=" * 80)
    print(f"{'Lang':<8} {'Model':<12} {'Layer':<6} {'Category':<10} {'Mean Dist':>10} {'Entropy':>9}")
    print("-" * 58)

    fam_agg = defaultdict(list)
    for key, m in all_metrics.items():
        k = (m['lang'], m['model_type'], m['layer_name'], m['category'])
        fam_agg[k].append((m['mean_distance_all_heads'], m['entropy_all_heads']))

    for lang in ['hindi', 'nepali']:
        for model_type in ['pretrained', 'finetuned']:
            for layer_name in ['early', 'late']:
                for cat in ['familiar', 'held_out']:
                    vals = fam_agg.get((lang, model_type, layer_name, cat), [])
                    if not vals:
                        continue
                    avg_d = float(np.mean([v[0] for v in vals]))
                    avg_e = float(np.mean([v[1] for v in vals]))
                    print(f"{lang:<8} {model_type:<12} {layer_name:<6} {cat:<10} {avg_d:>10.4f} {avg_e:>9.4f}")

    print()
    print("Phase 2 reference values (Layer 7, all heads, LM text):")
    for lang, ref in PHASE2_REFERENCE.items():
        print(f"  {lang}: mean_attn_distance={ref['mean_attention_distance']:.4f}  "
              f"entropy={ref['attention_entropy']:.4f}")

## Section 13 — Attention Summary Markdown

In [ ]:
if not RUN_ANALYSIS:
    print("RUN_ANALYSIS=False — skipping summary markdown.")
else:
    from pathlib import Path
    import numpy as np
    from collections import defaultdict

    md_lines = [
        "# Phase 3 Post-Finetuning Attention Analysis Summary",
        "",
        "## Methodology",
        "",
        "- **Models**: Pretrained (`step_00020000.pt`) vs. Fine-tuned (`step_00003000.pt`) "
        "for Hindi and Nepali.",
        "- **Layers analysed**: Layer 0 (early) and Layer 7 (late). "
        "Convention unchanged from Phase 2.",
        "- **Heads**: All 7 heads analysed. Head 0 used for heatmaps (Phase 2 continuity).",
        "- **Examples**: Deterministically selected from the TEST split "
        f"({N_EXAMPLES_PER_CATEGORY} familiar + {N_EXAMPLES_PER_CATEGORY} held-out per language).",
        "- **Identical inputs**: Pretrained and fine-tuned models receive exactly "
        "the same token IDs for each paired example.",
        "",
        "## Metric Definitions (Phase 2 definitions, reused exactly)",
        "",
        "- **Mean Attention Distance** = Σ_j p(i,j) × |i−j|, averaged over all query "
        "positions i and all heads.  ",
        "  A higher value indicates more **long-range** attention; "
        "a lower value indicates more **local** attention.",
        "- **Entropy** = −Σ_j p(i,j) × log(p(i,j)+ε), averaged over positions and heads.  ",
        "  A higher entropy = more diffuse/uniform attention; "
        "a lower entropy = sharper/more focused attention.",
        "",
        "## Table 1 — Aggregate Metrics by Language × Layer × Model",
        "",
        "| Language | Layer | Model | Mean Attn Distance | Entropy |",
        "|---|---|---|---|---|",
    ]

    agg = defaultdict(list)
    for key, m in all_metrics.items():
        agg_key = (m['lang'], m['layer_name'], m['model_type'])
        agg[agg_key].append((m['mean_distance_all_heads'], m['entropy_all_heads']))

    for lang in ['hindi', 'nepali']:
        for layer_name in ['early', 'late']:
            for model_type in ['pretrained', 'finetuned']:
                vals = agg.get((lang, layer_name, model_type), [])
                if not vals:
                    continue
                avg_dist = float(np.mean([v[0] for v in vals]))
                avg_ent  = float(np.mean([v[1] for v in vals]))
                md_lines.append(
                    f"| {lang} | {layer_name} | {model_type} | {avg_dist:.4f} | {avg_ent:.4f} |"
                )

    md_lines += [
        "",
        "## Table 2 — Head-Level Distance Change After Fine-tuning",
        "",
        "| Language | Layer | Head | Pre Distance | FT Distance | Δ Distance | Pre Entropy | FT Entropy | Δ Entropy |",
        "|---|---|---|---|---|---|---|---|---|",
    ]

    head_agg = defaultdict(lambda: defaultdict(list))
    for key, m in all_metrics.items():
        k = (m['lang'], m['layer_name'], m['model_type'])
        for h_data in m['per_head']:
            head_agg[k][h_data['head']].append((h_data['mean_distance'], h_data['entropy']))

    for lang in ['hindi', 'nepali']:
        for layer_name in ['early', 'late']:
            pre_key = (lang, layer_name, 'pretrained')
            ft_key  = (lang, layer_name, 'finetuned')
            for h in range(7):
                pre_v = head_agg.get(pre_key, {}).get(h, [])
                ft_v  = head_agg.get(ft_key,  {}).get(h, [])
                if not pre_v or not ft_v:
                    continue
                pre_d  = float(np.mean([v[0] for v in pre_v]))
                ft_d   = float(np.mean([v[0] for v in ft_v]))
                pre_e  = float(np.mean([v[1] for v in pre_v]))
                ft_e   = float(np.mean([v[1] for v in ft_v]))
                sign_d = f"+{ft_d-pre_d:.4f}" if ft_d >= pre_d else f"{ft_d-pre_d:.4f}"
                sign_e = f"+{ft_e-pre_e:.4f}" if ft_e >= pre_e else f"{ft_e-pre_e:.4f}"
                md_lines.append(
                    f"| {lang} | {layer_name} | {h} | {pre_d:.4f} | {ft_d:.4f} | {sign_d} "
                    f"| {pre_e:.4f} | {ft_e:.4f} | {sign_e} |"
                )

    md_lines += [
        "",
        "## Phase 2 Reference (for contextual comparison)",
        "",
        "Phase 2 values were computed on LM validation text, Layer 7, all heads:",
        "",
    ]
    for lang, ref in PHASE2_REFERENCE.items():
        md_lines.append(
            f"- **{lang.capitalize()}**: mean_attn_distance={ref['mean_attention_distance']:.4f}  "
            f"entropy={ref['attention_entropy']:.4f}"
        )

    md_lines += [
        "",
        "## Output Files",
        "",
        "- `attention_metrics.json` — full per-layer per-head metrics for all examples",
        "- `attention_metrics.csv`  — flat CSV for analysis",
        "- `selected_examples_metadata.json` — exact examples selected with token IDs",
        "- `figures/` — heatmap PNGs (8 files: lang × layer × model)",
    ]

    md_path = Path(OUTPUT_DIR) / "attention_summary.md"
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write("\n".join(md_lines))
    print(f"Saved: {md_path} ({md_path.stat().st_size} bytes)")
    print("\n" + "\n".join(md_lines[:40]))

## Section 14 — Final Assertions and File Verification

In [ ]:
if not RUN_ANALYSIS:
    print("RUN_ANALYSIS=False — skipping final assertions.")
else:
    from pathlib import Path
    print("=" * 60)
    print("FINAL ASSERTIONS")
    print("=" * 60)

    required_outputs = [
        Path(OUTPUT_DIR) / "attention_metrics.json",
        Path(OUTPUT_DIR) / "attention_metrics.csv",
        Path(OUTPUT_DIR) / "attention_summary.md",
        Path(OUTPUT_DIR) / "selected_examples_metadata.json",
        Path(OUTPUT_DIR) / "figures" / "hindi_early_pretrained.png",
        Path(OUTPUT_DIR) / "figures" / "hindi_early_finetuned.png",
        Path(OUTPUT_DIR) / "figures" / "hindi_late_pretrained.png",
        Path(OUTPUT_DIR) / "figures" / "hindi_late_finetuned.png",
        Path(OUTPUT_DIR) / "figures" / "nepali_early_pretrained.png",
        Path(OUTPUT_DIR) / "figures" / "nepali_early_finetuned.png",
        Path(OUTPUT_DIR) / "figures" / "nepali_late_pretrained.png",
        Path(OUTPUT_DIR) / "figures" / "nepali_late_finetuned.png",
    ]

    all_ok = True
    for fp in required_outputs:
        exists = fp.exists() and fp.stat().st_size > 0
        status = "✓" if exists else "✗ MISSING/EMPTY"
        print(f"  {status}  {fp.name}")
        if not exists:
            all_ok = False

    # Verify no raw tensors
    out = Path(OUTPUT_DIR)
    raw_tensor_files = list(out.rglob("*.npy")) + list(out.rglob("*.npz")) + list(out.rglob("*.pt"))
    assert len(raw_tensor_files) == 0, f"Raw tensor files found: {raw_tensor_files}"
    print("\n  ✓  No raw attention tensors on disk")

    # Verify attention shape properties
    for key, m in all_metrics.items():
        assert len(m['per_head']) == 7, f"Expected 7 heads, got {len(m['per_head'])}"
    print(f"  ✓  All {len(all_metrics)} metric records have exactly 7 heads")

    if all_ok:
        print(f"\n[PASS] All output files verified.")
    else:
        print(f"\n[FAIL] Some output files are missing. Check the analysis cells above.")

    print(f"\nAll results saved to: {OUTPUT_DIR}")
    print("\nPhase 3 attention analysis complete.")

## Done

Results written to `results/phase3_attention/`:
- `attention_metrics.json` — full metrics (per language × model × layer × head × example)
- `attention_metrics.csv`  — flat CSV suitable for further analysis
- `attention_summary.md`   — human-readable comparison tables
- `selected_examples_metadata.json` — exact examples used, with token IDs
- `figures/*.png`          — 8 paired heatmaps (lang × layer × model, Head 0)